# Module 3: RAG Systems Engineering - Hybrid Search & GraphRAG

This notebook is refactored into smaller implementation blocks so each stage is easy to run and debug independently.


## 1. Setup: Dependencies and Runtime Configuration

First load all dependencies used throughout ingestion and retrieval.


In [5]:
import math
import json
import os
import sys
from typing import List, Dict, Any, Tuple, Optional
from pathlib import Path
from dataclasses import dataclass
from collections import defaultdict

import requests
import networkx as nx
from pypdf import PdfReader
from rank_bm25 import BM25Okapi
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct
from dotenv import load_dotenv

# Add parent directory to path for logger_config
sys.path.append(str(Path.cwd().parent))
from logger_config import get_logger

# Set up logger
logger = get_logger(__name__)

print("✓ Imports successful")

✓ Imports successful


### Configuration Constants

Set service endpoints for OpenAI embeddings and Qdrant vector storage.


In [6]:
# Load environment variables from .env file in root directory
load_dotenv()
env = os.environ

# OpenAI embeddings configuration
OPENAI_API_URL = env.get("OPENAI_API_URL")
OPENAI_EMBED_MODEL = env.get("OPENAI_EMBED_MODEL")

# Qdrant connection
QDRANT_URL = env.get("QDRANT_URL")

print("✓ Runtime configuration set")
print("✓ OPENAI_API_KEY loaded" if env.get("OPENAI_API_KEY") else "⚠ OPENAI_API_KEY not found")
print("✓ QDRANT_URL loaded" if QDRANT_URL else "⚠ QDRANT_URL not found")

✓ Runtime configuration set
✓ OPENAI_API_KEY loaded
✓ QDRANT_URL loaded


## 2. Document Ingestion with Hierarchical Chunking

Define the core chunk schema that preserves section structure and metadata.


In [7]:
@dataclass
class DocumentChunk:
    """Represents a chunk of text from the policy document."""

    chunk_id: str
    text: str
    section_number: str  # e.g., "4.1", "9.2.1"
    section_title: str   # e.g., "Taxis and Rideshares"
    parent_section: Optional[str]  # e.g., "4" for "4.1"
    page_number: int
    hierarchy_path: List[str]  # e.g., ["Section 4", "4.1"]
    chunk_type: str  # "section", "subsection", "paragraph"

    def to_dict(self) -> Dict[str, Any]:
        return {
            "chunk_id": self.chunk_id,
            "text": self.text,
            "section_number": self.section_number,
            "section_title": self.section_title,
            "parent_section": self.parent_section,
            "page_number": self.page_number,
            "hierarchy_path": self.hierarchy_path,
            "chunk_type": self.chunk_type,
        }

print("✓ DocumentChunk defined")

✓ DocumentChunk defined


### Ingestion Pipeline

The pipeline loads PDF pages, detects sections/subsections, and emits hierarchical chunks.


In [8]:
class DocumentIngestionPipeline:
    """Ingests PDF documents with hierarchical chunking."""

    def __init__(self, pdf_path: str):
        self.pdf_path = Path(pdf_path)
        self.chunks: List[DocumentChunk] = []
        logger.info(f"Initialized ingestion pipeline for: {self.pdf_path}")

    def load_pdf(self) -> List[Tuple[int, str]]:
        """Load PDF and extract text by page."""
        logger.info(f"Loading PDF: {self.pdf_path}")
        reader = PdfReader(self.pdf_path)
        pages = []

        for page_num, page in enumerate(reader.pages, start=1):
            text = page.extract_text()
            pages.append((page_num, text))

        logger.info(f"Loaded {len(pages)} pages from {self.pdf_path.name}")
        return pages

    def _create_section_chunk(self, section_num: str, section_title: str, page_num: int, chunk_counter: int) -> DocumentChunk:
        """Create a section-level chunk."""
        return DocumentChunk(
            chunk_id=f"chunk_{chunk_counter}",
            text=f"Section {section_num}: {section_title}",
            section_number=section_num,
            section_title=section_title,
            parent_section=None,
            page_number=page_num,
            hierarchy_path=[f"Section {section_num}"],
            chunk_type="section"
        )

    def _create_subsection_chunk(self, subsection_num: str, subsection_title: str, current_section: str, page_num: int, chunk_counter: int) -> DocumentChunk:
        """Create a subsection-level chunk."""
        return DocumentChunk(
            chunk_id=f"chunk_{chunk_counter}",
            text=f"{subsection_num} {subsection_title}",
            section_number=subsection_num,
            section_title=subsection_title,
            parent_section=current_section,
            page_number=page_num,
            hierarchy_path=[f"Section {current_section}", subsection_num],
            chunk_type="subsection"
        )

    def _create_subsubsection_chunk(self, subsubsection_num: str, subsubsection_title: str, current_subsection: str, page_num: int, chunk_counter: int) -> DocumentChunk:
        """Create a sub-subsection-level chunk."""
        return DocumentChunk(
            chunk_id=f"chunk_{chunk_counter}",
            text=f"{subsubsection_num} {subsubsection_title}",
            section_number=subsubsection_num,
            section_title=subsubsection_title,
            parent_section=current_subsection,
            page_number=page_num,
            hierarchy_path=[f"Section {current_subsection.split('.')[0]}", current_subsection, subsubsection_num],
            chunk_type="subsubsection"
        )

    def _create_paragraph_chunk(self, para: str, current_section: str, current_subsection: str, page_num: int, chunk_counter: int) -> DocumentChunk:
        """Create a paragraph-level chunk."""
        parent = current_subsection or current_section
        hierarchy = []
        if current_section:
            hierarchy.append(f"Section {current_section}")
        if current_subsection:
            hierarchy.append(current_subsection)

        return DocumentChunk(
            chunk_id=f"chunk_{chunk_counter}",
            text=para,
            section_number=parent or "0",
            section_title="",
            parent_section=parent,
            page_number=page_num,
            hierarchy_path=hierarchy,
            chunk_type="paragraph"
        )

    def _process_paragraph(self, para: str, current_section: str, current_subsection: str, page_num: int, chunk_counter: int, chunks: List[DocumentChunk]) -> Tuple[str, str, int]:
        """Process a single paragraph and update tracking variables."""
        import re

        # Patterns for section detection
        section_pattern = re.compile(r'^Section (\d+):\s*(.+)$', re.MULTILINE)
        subsection_pattern = re.compile(r'^(\d+\.\d+)\s+(.+)$', re.MULTILINE)
        subsubsection_pattern = re.compile(r'^(\d+\.\d+\.\d+)\s+(.+)$', re.MULTILINE)

        # Check for main section
        section_match = section_pattern.search(para)
        if section_match:
            section_num = section_match.group(1)
            section_title = section_match.group(2).strip()
            chunks.append(self._create_section_chunk(section_num, section_title, page_num, chunk_counter))
            return section_num, None, chunk_counter + 1

        # Check for subsection (e.g., 4.1)
        subsection_match = subsection_pattern.search(para)
        if subsection_match and current_section:
            subsection_num = subsection_match.group(1)
            subsection_title = subsection_match.group(2).strip()
            chunks.append(self._create_subsection_chunk(subsection_num, subsection_title, current_section, page_num, chunk_counter))
            return current_section, subsection_num, chunk_counter + 1

        # Check for sub-subsection (e.g., 4.1.1)
        subsubsection_match = subsubsection_pattern.search(para)
        if subsubsection_match and current_subsection:
            subsubsection_num = subsubsection_match.group(1)
            subsubsection_title = subsubsection_match.group(2).strip()
            chunks.append(self._create_subsubsection_chunk(subsubsection_num, subsubsection_title, current_subsection, page_num, chunk_counter))
            return current_section, current_subsection, chunk_counter + 1

        # Regular paragraph - attach to current subsection or section
        if len(para) > 50:  # Only chunk substantial paragraphs
            chunks.append(self._create_paragraph_chunk(para, current_section, current_subsection, page_num, chunk_counter))
            return current_section, current_subsection, chunk_counter + 1

        return current_section, current_subsection, chunk_counter

    def detect_sections(self, pages: List[Tuple[int, str]]) -> List[DocumentChunk]:
        """Detect hierarchical sections and create chunks."""
        logger.info("Starting section detection and chunking")
        chunks = []
        current_section = None
        current_subsection = None
        chunk_counter = 0

        for page_num, page_text in pages:
            # Skip title page and TOC (first 3 pages)
            if page_num <= 3:
                continue

            # Split into paragraphs
            paragraphs = [p.strip() for p in page_text.split('\n\n') if p.strip()]

            for para in paragraphs:
                current_section, current_subsection, chunk_counter = self._process_paragraph(
                    para, current_section, current_subsection, page_num, chunk_counter, chunks
                )

        logger.info(f"Created {len(chunks)} hierarchical chunks")
        return chunks

    def ingest(self) -> List[DocumentChunk]:
        """Run the full ingestion pipeline."""
        logger.info("Starting full ingestion pipeline")
        pages = self.load_pdf()
        self.chunks = self.detect_sections(pages)
        logger.info(f"Ingestion pipeline completed with {len(self.chunks)} chunks")
        return self.chunks

print("✓ DocumentIngestionPipeline defined")

✓ DocumentIngestionPipeline defined


### Run Ingestion

Ingest the sample policy and inspect a few chunk examples.


In [9]:
# Ingest the policy document
pipeline = DocumentIngestionPipeline("sample_expense_policy.pdf")
chunks = pipeline.ingest()

# Show some example chunks
print("\n=== Example Chunks ===")
for i, chunk in enumerate(chunks[:5]):
    print(f"\nChunk {i+1}:")
    print(f"  ID: {chunk.chunk_id}")
    print(f"  Section: {chunk.section_number} - {chunk.section_title}")
    print(f"  Parent: {chunk.parent_section}")
    print(f"  Page: {chunk.page_number}")
    print(f"  Hierarchy: {' > '.join(chunk.hierarchy_path)}")
    print(f"  Type: {chunk.chunk_type}")
    print(f"  Text preview: {chunk.text[:100]}...")

print(f"\n✓ Total chunks created: {len(chunks)}")

2026-03-01 12:26:08 - __main__ - INFO - Initialized ingestion pipeline for: sample_expense_policy.pdf
2026-03-01 12:26:08 - __main__ - INFO - Starting full ingestion pipeline
2026-03-01 12:26:08 - __main__ - INFO - Loading PDF: sample_expense_policy.pdf
incorrect startxref pointer(1)
parsing for Object Streams
2026-03-01 12:26:08 - __main__ - INFO - Loaded 17 pages from sample_expense_policy.pdf
2026-03-01 12:26:08 - __main__ - INFO - Starting section detection and chunking
2026-03-01 12:26:08 - __main__ - INFO - Created 14 hierarchical chunks
2026-03-01 12:26:08 - __main__ - INFO - Ingestion pipeline completed with 14 chunks



=== Example Chunks ===

Chunk 1:
  ID: chunk_0
  Section: 2 - Meals & Entertainment
  Parent: None
  Page: 4
  Hierarchy: Section 2
  Type: section
  Text preview: Section 2: Meals & Entertainment...

Chunk 2:
  ID: chunk_1
  Section: 2.2 - Client Dinners and Business Meals
  Parent: 2
  Page: 5
  Hierarchy: Section 2 > 2.2
  Type: subsection
  Text preview: 2.2 Client Dinners and Business Meals...

Chunk 3:
  ID: chunk_2
  Section: 2.3 - Alcohol Policy
  Parent: 2
  Page: 6
  Hierarchy: Section 2 > 2.3
  Type: subsection
  Text preview: 2.3 Alcohol Policy...

Chunk 4:
  ID: chunk_3
  Section: 3 - Lodging and Accommodations
  Parent: None
  Page: 7
  Hierarchy: Section 3
  Type: section
  Text preview: Section 3: Lodging and Accommodations...

Chunk 5:
  ID: chunk_4
  Section: 4 - Ground Transport
  Parent: None
  Page: 8
  Hierarchy: Section 4
  Type: section
  Text preview: Section 4: Ground Transport...

✓ Total chunks created: 14


## 3. Dense Vector Retrieval (Semantic Search)

Dense retrieval captures semantic similarity using embeddings and Qdrant.


In [10]:
class DenseVectorRetriever:
    """Dense vector search using OpenAI embeddings + Qdrant."""

    def __init__(self, collection_name: str = "policy_chunks"):
        self.collection_name = collection_name
        self.embed_model = OPENAI_EMBED_MODEL
        self.embedding_dim = 1536  # OpenAI text-embedding-3-small dimension
        self.client = QdrantClient(url=QDRANT_URL)
        logger.info(f"Initialized DenseVectorRetriever with collection: {collection_name}")

    def _get_embedding(self, text: str) -> List[float]:
        """Get OpenAI embedding for a single text."""
        logger.debug(f"Getting embedding for text: {text[:50]}...")
        response = requests.post(
            f"{OPENAI_API_URL}/embeddings",
            headers={"Authorization": f"Bearer {env.get('OPENAI_API_KEY')}"},
            json={"model": self.embed_model, "input": text},
            timeout=30,
        )
        response.raise_for_status()
        embedding = response.json()["data"][0]["embedding"]
        logger.debug(f"Embedding generated successfully (dim: {len(embedding)})")
        return embedding

    def create_collection(self):
        """Create Qdrant collection for policy chunks."""
        logger.info(f"Creating Qdrant collection: {self.collection_name}")
        try:
            self.client.delete_collection(self.collection_name)
        except Exception:
            # Collection might not exist, which is fine for creation
            pass

        self.client.create_collection(
            collection_name=self.collection_name,
            vectors_config=VectorParams(
                size=self.embedding_dim,
                distance=Distance.COSINE
            )
        )
        logger.info(f"Created Qdrant collection: {self.collection_name}")

    def index_chunks(self, chunks: List[DocumentChunk]):
        """Index document chunks in Qdrant."""
        logger.info(f"Indexing {len(chunks)} chunks...")
        
        points = []
        for i, chunk in enumerate(chunks):
            if i % 10 == 0:
                logger.debug(f"Embedding chunk {i+1}/{len(chunks)}...")
            
            embedding = self._get_embedding(chunk.text)
            point = PointStruct(
                id=i,
                vector=embedding,
                payload=chunk.to_dict()
            )
            points.append(point)
        
        self.client.upsert(
            collection_name=self.collection_name,
            points=points
        )
        logger.info(f"Indexed {len(points)} chunks in Qdrant")

    def search(self, query: str, top_k: int = 5) -> List[Tuple[DocumentChunk, float]]:
        """Semantic search for relevant chunks."""
        logger.info(f"Performing semantic search for: {query[:50]}...")
        query_embedding = self._get_embedding(query)

        results = self.client.query_points(
            collection_name=self.collection_name,
            query=query_embedding,
            limit=top_k
        ).points

        retrieved = []
        for result in results:
            chunk = DocumentChunk(**result.payload)
            score = result.score
            retrieved.append((chunk, score))
        
        logger.info(f"Found {len(retrieved)} results for query")
        return retrieved

print("✓ DenseVectorRetriever defined")

✓ DenseVectorRetriever defined


### Build Dense Index

Create the vector collection and upload chunk embeddings.


In [11]:
# Create dense retriever and index chunks
dense_retriever = DenseVectorRetriever()
dense_retriever.create_collection()
dense_retriever.index_chunks(chunks)

print("\n✓ Dense vector indexing complete")

2026-03-01 12:26:10 - __main__ - INFO - Initialized DenseVectorRetriever with collection: policy_chunks
2026-03-01 12:26:10 - __main__ - INFO - Creating Qdrant collection: policy_chunks
2026-03-01 12:26:13 - __main__ - INFO - Created Qdrant collection: policy_chunks
2026-03-01 12:26:13 - __main__ - INFO - Indexing 14 chunks...
2026-03-01 12:26:27 - __main__ - INFO - Indexed 14 chunks in Qdrant



✓ Dense vector indexing complete


### Test Dense Search

Run semantic queries and inspect top results.


In [12]:
# Test semantic search
test_queries = [
    "What is the meal limit for individual employees?",
    "Can I expense sustenance while traveling?",
    "What are the rules for client dinners?",
]

print("=== Dense Vector Search Results ===")
for query in test_queries:
    print(f"\nQuery: {query}")
    results = dense_retriever.search(query, top_k=3)

    for i, (chunk, score) in enumerate(results, 1):
        print(f"\n  Result {i} (score: {score:.3f}):")
        print(f"    Section: {chunk.section_number} - {chunk.section_title}")
        print(f"    Page: {chunk.page_number}")
        print(f"    Text: {chunk.text[:150]}...")

2026-03-01 12:26:27 - __main__ - INFO - Performing semantic search for: What is the meal limit for individual employees?...


=== Dense Vector Search Results ===

Query: What is the meal limit for individual employees?


2026-03-01 12:26:27 - __main__ - INFO - Found 3 results for query
2026-03-01 12:26:27 - __main__ - INFO - Performing semantic search for: Can I expense sustenance while traveling?...



  Result 1 (score: 0.458):
    Section: 2.2 - Client Dinners and Business Meals
    Page: 5
    Text: 2.2 Client Dinners and Business Meals...

  Result 2 (score: 0.426):
    Section: 9.2 - VP and Above - Meal and Entertainment
    Page: 16
    Text: 9.2 VP and Above - Meal and Entertainment...

  Result 3 (score: 0.393):
    Section: 2 - Meals & Entertainment
    Page: 4
    Text: Section 2: Meals & Entertainment...

Query: Can I expense sustenance while traveling?


2026-03-01 12:26:29 - __main__ - INFO - Found 3 results for query
2026-03-01 12:26:29 - __main__ - INFO - Performing semantic search for: What are the rules for client dinners?...



  Result 1 (score: 0.420):
    Section: 8 - Miscellaneous Expenses
    Page: 14
    Text: Section 8: Miscellaneous Expenses...

  Result 2 (score: 0.390):
    Section: 9.2 - VP and Above - Meal and Entertainment
    Page: 16
    Text: 9.2 VP and Above - Meal and Entertainment...

  Result 3 (score: 0.389):
    Section: 2 - Meals & Entertainment
    Page: 4
    Text: Section 2: Meals & Entertainment...

Query: What are the rules for client dinners?


2026-03-01 12:26:30 - __main__ - INFO - Found 3 results for query



  Result 1 (score: 0.721):
    Section: 2.2 - Client Dinners and Business Meals
    Page: 5
    Text: 2.2 Client Dinners and Business Meals...

  Result 2 (score: 0.472):
    Section: 9.2 - VP and Above - Meal and Entertainment
    Page: 16
    Text: 9.2 VP and Above - Meal and Entertainment...

  Result 3 (score: 0.398):
    Section: 2 - Meals & Entertainment
    Page: 4
    Text: Section 2: Meals & Entertainment...


## 4. Sparse BM25 Retrieval (Keyword Search)

Sparse retrieval is effective for exact codes, amounts, and phrase matching.


In [13]:
class SparseBM25Retriever:
    """Sparse keyword retrieval using BM25."""

    def __init__(self):
        self.chunks: List[DocumentChunk] = []
        self.bm25: Optional[BM25Okapi] = None
        self.tokenized_corpus: List[List[str]] = []

    def _tokenize(self, text: str) -> List[str]:
        """Simple tokenization: lowercase and split on whitespace."""
        return text.lower().split()

    def index_chunks(self, chunks: List[DocumentChunk]):
        """Build BM25 index over chunks."""
        self.chunks = chunks
        self.tokenized_corpus = [self._tokenize(chunk.text) for chunk in chunks]
        self.bm25 = BM25Okapi(self.tokenized_corpus)
        print(f"✓ Built BM25 index over {len(chunks)} chunks")

    def search(self, query: str, top_k: int = 5) -> List[Tuple[DocumentChunk, float]]:
        """Keyword search for relevant chunks."""
        if not self.bm25:
            raise ValueError("BM25 index not built. Call index_chunks() first.")

        tokenized_query = self._tokenize(query)
        scores = self.bm25.get_scores(tokenized_query)
        top_indices = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)[:top_k]

        results = [(self.chunks[i], scores[i]) for i in top_indices]
        return results

print("✓ SparseBM25Retriever defined")


✓ SparseBM25Retriever defined


### Build and Test BM25 Index

Run keyword-heavy queries including policy codes and dollar limits.


In [14]:
# Create BM25 retriever and index chunks
bm25_retriever = SparseBM25Retriever()
bm25_retriever.index_chunks(chunks)

# Test keyword search
test_queries = [
    "PROJ-2024-001",
    "$150 client dinner",
    "VP first class",
]

print("\n=== BM25 Keyword Search Results ===")
for query in test_queries:
    print(f"\nQuery: {query}")
    results = bm25_retriever.search(query, top_k=3)

    for i, (chunk, score) in enumerate(results, 1):
        print(f"\n  Result {i} (score: {score:.3f}):")
        print(f"    Section: {chunk.section_number} - {chunk.section_title}")
        print(f"    Page: {chunk.page_number}")
        print(f"    Text: {chunk.text[:150]}...")

✓ Built BM25 index over 14 chunks

=== BM25 Keyword Search Results ===

Query: PROJ-2024-001

  Result 1 (score: 0.000):
    Section: 2 - Meals & Entertainment
    Page: 4
    Text: Section 2: Meals & Entertainment...

  Result 2 (score: 0.000):
    Section: 2.2 - Client Dinners and Business Meals
    Page: 5
    Text: 2.2 Client Dinners and Business Meals...

  Result 3 (score: 0.000):
    Section: 2.3 - Alcohol Policy
    Page: 6
    Text: 2.3 Alcohol Policy...

Query: $150 client dinner

  Result 1 (score: 1.926):
    Section: 2.2 - Client Dinners and Business Meals
    Page: 5
    Text: 2.2 Client Dinners and Business Meals...

  Result 2 (score: 0.000):
    Section: 2 - Meals & Entertainment
    Page: 4
    Text: Section 2: Meals & Entertainment...

  Result 3 (score: 0.000):
    Section: 2.3 - Alcohol Policy
    Page: 6
    Text: 2.3 Alcohol Policy...

Query: VP first class

  Result 1 (score: 1.643):
    Section: 9.2 - VP and Above - Meal and Entertainment
    Page: 16
    Text:

## 5. Hybrid Search: Dense + Sparse Fusion

Hybrid retrieval combines semantic recall and exact-match precision using Reciprocal Rank Fusion (RRF).


In [15]:
class HybridRetriever:
    """Hybrid retrieval combining dense vector and sparse BM25 search."""

    def __init__(
        self,
        dense_retriever: DenseVectorRetriever,
        sparse_retriever: SparseBM25Retriever,
        dense_weight: float = 0.5,
        sparse_weight: float = 0.5,
    ):
        self.dense_retriever = dense_retriever
        self.sparse_retriever = sparse_retriever
        self.dense_weight = dense_weight
        self.sparse_weight = sparse_weight

    def reciprocal_rank_fusion(
        self,
        dense_results: List[Tuple[DocumentChunk, float]],
        sparse_results: List[Tuple[DocumentChunk, float]],
        k: int = 60,
    ) -> List[Tuple[DocumentChunk, float]]:
        """Merge results using Reciprocal Rank Fusion.

        RRF score = sum(1 / (k + rank)) for each retriever where doc appears.
        """
        scores = defaultdict(float)
        chunk_map = {}

        for rank, (chunk, _) in enumerate(dense_results, start=1):
            rrf_score = self.dense_weight / (k + rank)
            scores[chunk.chunk_id] += rrf_score
            chunk_map[chunk.chunk_id] = chunk

        for rank, (chunk, _) in enumerate(sparse_results, start=1):
            rrf_score = self.sparse_weight / (k + rank)
            scores[chunk.chunk_id] += rrf_score
            chunk_map[chunk.chunk_id] = chunk

        sorted_chunks = sorted(
            scores.items(),
            key=lambda x: x[1],
            reverse=True
        )

        return [(chunk_map[chunk_id], score) for chunk_id, score in sorted_chunks]

    def search(self, query: str, top_k: int = 5) -> List[Tuple[DocumentChunk, float]]:
        """Hybrid search combining dense and sparse retrieval."""
        dense_results = self.dense_retriever.search(query, top_k=top_k * 2)
        sparse_results = self.sparse_retriever.search(query, top_k=top_k * 2)
        hybrid_results = self.reciprocal_rank_fusion(dense_results, sparse_results)
        return hybrid_results[:top_k]

print("✓ HybridRetriever defined")

✓ HybridRetriever defined


### Test Hybrid Search

Run mixed semantic + keyword queries and inspect fused ranking.


In [16]:
# Create hybrid retriever
hybrid_retriever = HybridRetriever(
    dense_retriever=dense_retriever,
    sparse_retriever=bm25_retriever,
    dense_weight=0.6,  # Slightly favor semantic search
    sparse_weight=0.4,
)

# Test hybrid search
test_queries = [
    "What is the meal allowance for sustenance?",  # Semantic query
    "PROJ-2024-002 client development",  # Keyword query
    "Can VP fly first class?",  # Mixed query
]

print("=== Hybrid Search Results ===")
for query in test_queries:
    print(f"\n{'='*60}")
    print(f"Query: {query}")
    print(f"{'='*60}")

    results = hybrid_retriever.search(query, top_k=3)

    for i, (chunk, score) in enumerate(results, 1):
        print(f"\nResult {i} (RRF score: {score:.4f}):")
        print(f"  Section: {chunk.section_number} - {chunk.section_title}")
        print(f"  Page: {chunk.page_number}")
        print(f"  Hierarchy: {' > '.join(chunk.hierarchy_path)}")
        print(f"  Text: {chunk.text[:200]}...")

2026-03-01 12:26:30 - __main__ - INFO - Performing semantic search for: What is the meal allowance for sustenance?...


=== Hybrid Search Results ===

Query: What is the meal allowance for sustenance?


2026-03-01 12:26:32 - __main__ - INFO - Found 6 results for query
2026-03-01 12:26:32 - __main__ - INFO - Performing semantic search for: PROJ-2024-002 client development...



Result 1 (RRF score: 0.0163):
  Section: 2 - Meals & Entertainment
  Page: 4
  Hierarchy: Section 2
  Text: Section 2: Meals & Entertainment...

Result 2 (RRF score: 0.0162):
  Section: 9.2 - VP and Above - Meal and Entertainment
  Page: 16
  Hierarchy: Section 9 > 9.2
  Text: 9.2 VP and Above - Meal and Entertainment...

Result 3 (RRF score: 0.0159):
  Section: 2.2 - Client Dinners and Business Meals
  Page: 5
  Hierarchy: Section 2 > 2.2
  Text: 2.2 Client Dinners and Business Meals...

Query: PROJ-2024-002 client development


2026-03-01 12:26:33 - __main__ - INFO - Found 6 results for query
2026-03-01 12:26:33 - __main__ - INFO - Performing semantic search for: Can VP fly first class?...



Result 1 (RRF score: 0.0164):
  Section: 2.2 - Client Dinners and Business Meals
  Page: 5
  Hierarchy: Section 2 > 2.2
  Text: 2.2 Client Dinners and Business Meals...

Result 2 (RRF score: 0.0157):
  Section: 2.3 - Alcohol Policy
  Page: 6
  Hierarchy: Section 2 > 2.3
  Text: 2.3 Alcohol Policy...

Result 3 (RRF score: 0.0154):
  Section: 4 - Ground Transport
  Page: 8
  Hierarchy: Section 4
  Text: Section 4: Ground Transport...

Query: Can VP fly first class?


2026-03-01 12:26:34 - __main__ - INFO - Found 6 results for query



Result 1 (RRF score: 0.0164):
  Section: 9.2 - VP and Above - Meal and Entertainment
  Page: 16
  Hierarchy: Section 9 > 9.2
  Text: 9.2 VP and Above - Meal and Entertainment...

Result 2 (RRF score: 0.0154):
  Section: 2.2 - Client Dinners and Business Meals
  Page: 5
  Hierarchy: Section 2 > 2.2
  Text: 2.2 Client Dinners and Business Meals...

Result 3 (RRF score: 0.0153):
  Section: 4 - Ground Transport
  Page: 8
  Hierarchy: Section 4
  Text: Section 4: Ground Transport...


## Next Step

If you want to continue, add GraphRAG entity/relationship extraction and policy exception traversal as the next section.
